In [1]:
import pandas as pd

# --- UPDATE THESE PATHS TO MATCH YOUR LOCAL FILES ---
FILES = {
    'main':  '2016-2026-pse-attacks-on-health-care-incident-data.xlsx',
    'shcc':  '2023-2024-pse-shcc-health-care-data (1).xlsx',
    'insec': 'InsecurityInsight_AttacksonHealthcareOPT (shared on 06.03.26).xlsx',
}

SGBV_COL = 'Health Workers Sexually Assaulted'
DATE_COL = 'Date'
DESC_COL = 'Event Description'
ID_COL   = 'SiND Event ID'


In [3]:
import pandas as pd

FILES = {
    'main':  '2016-2026-pse-attacks-on-health-care-incident-data.xlsx',
    'shcc':  '2023-2024-pse-shcc-health-care-data.xlsx',
    'insec': 'InsecurityInsight_AttacksonHealthcareOPT (shared on 06.03.26).xlsx',
}

SGBV_COL = 'Health Workers Sexually Assaulted'
DATE_COL = 'Date'
DESC_COL = 'Event Description'
ID_COL   = 'SiND Event ID'

def excel_serial_to_date(serial):
    try:
        return (pd.Timestamp('1899-12-30') + pd.Timedelta(days=float(serial))).date()
    except:
        return None

dfs = {}
for name, path in FILES.items():
    sheet = 'Data' if name == 'insec' else (0 if name == 'main' else '2023-2024 PSE SHCC Health Care ')
    df = pd.read_excel(path, sheet_name=sheet, header=0)
    df = df[df[DATE_COL].astype(str) != '#date'].copy()
    df[DATE_COL] = pd.to_numeric(df[DATE_COL], errors='coerce')
    df[DATE_COL] = df[DATE_COL].apply(excel_serial_to_date)
    df[SGBV_COL] = pd.to_numeric(df[SGBV_COL], errors='coerce')
    dfs[name] = df
    print(f'{name}: {len(df)} rows loaded')

main: 4218 rows loaded
shcc: 2531 rows loaded
insec: 3104 rows loaded


In [4]:
# Find all rows where SGBV = 44 across all files
cols_to_show = [DATE_COL, ID_COL, SGBV_COL, 'Health Workers Killed', 'Health Workers Injured',
                'Health Workers Arrested', 'Reported Perpetrator Name', 'Location of Incident']

for name, df in dfs.items():
    hits = df[df[SGBV_COL] == 44]
    available_cols = [c for c in cols_to_show if c in df.columns]
    print(f'\n=== {name.upper()} — {len(hits)} row(s) with SGBV=44 ===')
    if len(hits):
        display(hits[available_cols])



=== MAIN — 1 row(s) with SGBV=44 ===


,Date,SiND Event ID,Health Workers Sexually Assaulted,Health Workers Killed,Health Workers Injured,Health Workers Arrested,Reported Perpetrator Name,Location of Incident
814,None,84911,44,0,4,44,Israeli Defence Forces,Health Building



=== SHCC — 1 row(s) with SGBV=44 ===


,Date,SiND Event ID,Health Workers Sexually Assaulted,Health Workers Killed,Health Workers Injured,Health Workers Arrested,Reported Perpetrator Name
207,None,84911,44,0,4,44,Israeli Defence Forces



=== INSEC — 1 row(s) with SGBV=44 ===


,Date,SiND Event ID,Health Workers Sexually Assaulted,Health Workers Killed,Health Workers Injured,Health Workers Arrested,Reported Perpetrator Name,Location of Incident
800,None,84911,44,0,4,44,Israeli Defence Forces,Health Building


In [5]:
# Print full event description for any matching rows in the Insecurity Insight file
hits_insec = dfs['insec'][dfs['insec'][SGBV_COL] == 44]

if len(hits_insec):
    for _, row in hits_insec.iterrows():
        print(f"Date:        {row[DATE_COL]}")
        print(f"SiND ID:     {row[ID_COL]}")
        print(f"Perpetrator: {row.get('Reported Perpetrator Name', 'N/A')}")
        print(f"Location:    {row.get('Location of Incident', 'N/A')}")
        print(f"SGBV count:  {row[SGBV_COL]}")
        print(f"Description: {row.get(DESC_COL, 'N/A')}")
        print('-' * 80)
else:
    print('No SGBV=44 rows found in Insecurity Insight file.')
    print('Checking SHCC file for description...')
    hits_shcc = dfs['shcc'][dfs['shcc'][SGBV_COL] == 44]
    display(hits_shcc)


Date:        None
SiND ID:     84911
Perpetrator: Israeli Defence Forces
Location:    Health Building
SGBV count:  44
Description: 25 October 2024: A medical facility was raided by Israeli forces who opened fire inside. Three nurses and a cleaner were injured. Patients and staff were held without food or water and an all men, including 44 medical staff, were detained after being stripped; an INGO nursing director and INGO surgeon were among them.The medical facility director and 13 other staff were later released. The medical facility perimeter, three ambulances, a transport vehicle, medical supplies and its solar electric system were destroyed. When IDF withdrew the next day, only one paediatrician remained in the facility.
--------------------------------------------------------------------------------


In [6]:
hit = dfs['insec'][dfs['insec'][ID_COL] == 84911]

for _, row in hit.iterrows():
    for col in hit.columns:
        val = row[col]
        if pd.notna(val) and str(val) not in ['0', '0.0', 'nan', 'False', 'false', 'NotApplicable']:
            print(f"{col}: {val}")

Event Description: 25 October 2024: A medical facility was raided by Israeli forces who opened fire inside. Three nurses and a cleaner were injured. Patients and staff were held without food or water and an all men, including 44 medical staff, were detained after being stripped; an INGO nursing director and INGO surgeon were among them.The medical facility director and 13 other staff were later released. The medical facility perimeter, three ambulances, a transport vehicle, medical supplies and its solar electric system were destroyed. When IDF withdrew the next day, only one paediatrician remained in the facility.
Country: OPT
Country ISO: PSE
Admin 1: Gaza Strip
Latitude: 31.5
Longitude: 34.5
Geo Precision: (2) 25 km Precision 
Reported Perpetrator: Government: Military
Reported Perpetrator Name: Israeli Defence Forces
Weapon Carried/Used: Firearms
Location of Incident: Health Building
Number of Attacks on Health Facilities Reporting Damaged: 1
Forceful Entry into Health Facility: 1
